In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Quantum Support Vector Machines — QML: Classical and Quantum SVM Kernels with Tensor-Network Scaling

$\renewcommand{\ket}[1]{|#1\rangle}\renewcommand{\bra}[1]{\langle#1|}\newcommand{\braket}[2]{\langle#1|#2\rangle}$

**What You Will Do:**
* Compare linear, polynomial, and radial basis function (RBF) support vector machines on a nonlinear dataset
* Build a quantum fidelity SVM kernel with deterministic CUDA-Q state overlaps
* Test how data re-uploading and entangling connectivity change a quantum feature map
* Evaluate an entangling 36-qubit quantum fidelity SVM kernel with the CUDA-Q `tensornet` backend
* Distinguish a backend-scaling demonstration from evidence of model generalization or quantum advantage

**Prerequisites:**
* Python, NumPy, and Jupyter familiarity
* Vectors and dot products
* Basic quantum computing concepts: qubits, single-qubit rotations, controlled gates, and measurement
* No prior knowledge of support vector machines or the kernel trick is required

**Key Terminology:**
* Support Vector Machine (SVM)
* Support vector
* SVM kernel function
* SVM kernel matrix
* Quantum Support Vector Machine (QSVM)
* Quantum feature map
* Positive-semidefinite SVM kernel
* Data re-uploading

**CUDA-Q Syntax:**
* [`cudaq.kernel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.kernel) — defines a CUDA-Q quantum program (also called a CUDA-Q kernel)
* [`cudaq.qvector`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.qvector) — allocates a register of qubits
* [`cudaq.get_state`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.get_state) — returns a simulated quantum state
* [`cudaq.set_target`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.set_target) — selects a simulator or hardware backend
* [`cudaq.get_target`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html) — returns the active CUDA-Q target
* [`State.overlap`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html) — computes the inner product with another state
* [`State.amplitude`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html) — returns a selected computational-basis amplitude
**Exercises:** Complete each TODO region, then compare your work with the solution notebook.

**Solutions:** [solutions/03_quantum_svm_solutions.ipynb](solutions/03_quantum_svm_solutions.ipynb)

Try the hosted [QSVM Margin Explorer](https://nvidia.github.io/cuda-q-academic/quantum-machine-learning-and-data-analysis/interactive_widget/qsvm.html) to explore decision boundaries, margins, and support vectors.


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 12px 15px 12px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900;">&#9889; GPU Required:</span>** Parts of this notebook require a GPU. Section 3 is classical, Sections 4–5 use CUDA-Q's `nvidia` state-vector target, and Section 6 uses the GPU-only `tensornet` target. The small circuits can instead use `qpp-cpu`, but the 36-qubit example exceeds the memory of the lesson GPU as a dense state vector.

</div>

In [ ]:
## Instructions for Google Colab. You can ignore this cell if you have CUDA-Q
## set up locally with all required files on your system.
## Uncomment the lines below and execute this cell to install CUDA-Q.

#!pip install cudaq -q
#!pip install -q numpy matplotlib scikit-learn
#
#!wget -q https://github.com/NVIDIA/cuda-q-academic/archive/refs/heads/main.zip
#!unzip -q main.zip
#!mv cuda-q-academic-main/quantum-machine-learning-and-data-analysis/images ./images


> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).


In [ ]:
# Standard library
import time

# Scientific computing
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# CUDA-Q
import cudaq

# Machine learning
from sklearn.datasets import load_digits, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

## 1. The Classification Problem

In the previous QML notebooks, you explored hybrid quantum–classical neural networks for supervised learning. This lesson studies a different approach: the **Support Vector Machine (SVM)**, a classifier that chooses a **decision boundary** with as much separation as possible between two classes. A **Quantum Support Vector Machine (QSVM)** keeps the classical SVM training algorithm but uses a quantum circuit to calculate an **SVM kernel function**—a pairwise similarity used by the classifier. This is distinct from a **CUDA-Q kernel**, which is a quantum program defined with `@cudaq.kernel`. You will first build the classical picture from the ground up, then compute quantum fidelity SVM kernels with CUDA-Q programs, and finally explore a 36-qubit tensor-network example. 

Suppose we have examples from two known classes and want to classify future examples. The first task is to learn a decision boundary: a rule that divides feature space into a region predicted as one class and a region predicted as the other. 

![Two irregular wave-like data bands for which a straight boundary fails, an overly flexible curved boundary works, and a smoother nonlinear boundary follows the broad separating channel](images/svm_classification_boundaries.svg)

In the example above, each panel shows the same training data. A straight line cannot separate data with this shape. Both nonlinear boundaries separate the observed points, but the smoother one follows the broad channel between the classes instead of reacting to every local variation. Section 2 explains how an SVM chooses a good boundary.

## 2. Classical SVM Foundations

This section will explain the basics of how an SVM works, and build towards examples where you can build your own classical SVM implementations.



### From Feature Vectors to a Decision Boundary

The $i$ th **training example** is $(\mathbf{x}_i,y_i)$, where the **feature vector** $\mathbf{x}_i\in\mathbb{R}^n$ contains $n$ numerical features and the label $y_i\in\{-1,+1\}$ identifies its class. The derivation uses labels $-1$ and $+1$; later code uses 0 and 1.

During training, the SVM learns the weight vector $\mathbf{w}$ and offset $b$ from the labeled training examples. These are the model parameters: $\mathbf{w}$ determines the boundary's orientation and $b$ determines its position. A linear SVM then assigns an input $\mathbf{x}$ the score

$$g(\mathbf{x})=\mathbf{w}\cdot\mathbf{x}+b.$$

The sign of $g(\mathbf{x})$ gives the predicted class (see figure below). The set $g(\mathbf{x})=0$ is the **decision boundary**: a line in 2D, a plane in 3D, and a hyperplane in $n$ dimensions. The learned weight vector $\mathbf{w}$ is perpendicular to that boundary: for any two boundary points $\mathbf{x}_a$ and $\mathbf{x}_b$, subtracting their score equations gives $\mathbf{w}\cdot(\mathbf{x}_a-\mathbf{x}_b)=0$. Thus $\mathbf{w}$ points toward increasing score, while $b$ shifts the boundary.

![A linear SVM boundary separating negative and positive score regions, with two example predictions and the normal vector w](images/svm_decision_score.svg)

*The boundary is where the score changes sign; $\mathbf{w}$ points toward larger scores.*





### Margin and Support Vectors

Many boundaries may classify every training example correctly. A **hard-margin SVM** chooses the one with the widest empty band between the two classes. The closest training examples touch the edges of that band and determine the fitted boundary; these are the **support vectors**. Examples farther away do not affect the hard-margin solution.

![A maximum-margin SVM boundary with two dashed margin lines and one support vector from each class](images/svm_margin_distance_support.svg)

*The ringed support vectors touch the margins; moving either one can move the fitted boundary.*




### The Optimization Problem

The numerical scale of $g(\mathbf{x})=\mathbf{w}\cdot\mathbf{x}+b$ is otherwise arbitrary: rescaling $\mathbf{w}$ and $b$ together changes the score values but not the boundary or predicted signs. The conventional hard-margin score scale fixes the nearest correctly classified examples at signed score 1. On this common scale, maximizing the geometric margin is equivalent to minimizing $\lVert\mathbf{w}\rVert$. The hard-margin **primal problem** is therefore

$$\boxed{\min_{\mathbf{w},b}\;\frac12\lVert\mathbf{w}\rVert^2\quad\text{subject to}\quad y_i(\mathbf{w}\cdot\mathbf{x}_i+b)\geq1\;\;\forall i.}$$


<details>
<summary><strong>Optional: Deriving the margin–weight relationship</strong></summary>

For labels $y_i\in\{-1,+1\}$, the signed geometric distance from training example $i$ to the boundary is

$$d_i=\frac{y_i(\mathbf{w}\cdot\mathbf{x}_i+b)}{\lVert\mathbf{w}\rVert}.$$

The hard-margin half-width is $m=\min_i d_i$. After fixing the nearest signed score at 1, $m=1/\lVert\mathbf{w}\rVert$ and the full margin width is $2/\lVert\mathbf{w}\rVert$. This is why minimizing $\tfrac12\lVert\mathbf{w}\rVert^2$ maximizes the margin.

![Two canonical score slopes showing that a smaller weight norm reaches score one at a larger distance](images/svm_weight_norm_margin.svg)

</details>



### From Hard Margin to Soft Margin

Real datasets often overlap or contain noise. A **soft-margin** SVM introduces slack variables $\xi_i\geq0$ so examples may enter the margin or cross the boundary, while $C$ controls the penalty:

$$\boxed{\min_{\mathbf{w},b,\boldsymbol{\xi}}\;\frac12\lVert\mathbf{w}\rVert^2+C\sum_i\xi_i\quad\text{subject to}\quad y_i(\mathbf{w}\cdot\mathbf{x}_i+b)\geq1-\xi_i,\;\xi_i\geq0.}$$

Larger $C$ makes violations more costly; smaller $C$ allows more violations in exchange for stronger regularization. Unlike the separable hard-margin normalization, a soft-margin solution may have signed scores below 1 or even below 0. Soft-margin support vectors may lie on the margin, inside it, or on the wrong side.

![Hard-margin data with no violations beside soft-margin data with points inside the margin and on the wrong side](images/svm_hard_vs_soft_margin.svg)

*All classical and quantum examples later in this notebook use soft-margin `SVC` with $C=1$.*



### The Dual: The Same Training Problem in Pairwise Form

The primal problems above learn the boundary parameters $\mathbf{w}$ and $b$ directly. The **dual problem** is an equivalent reformulation of the same SVM training problem: it produces the same fitted boundary, but describes that boundary in terms of the training examples.

In the dual formulation, each training example receives a nonnegative coefficient $\alpha_i$. These coefficients arise by assigning a **Lagrange multiplier** to each margin constraint—a bookkeeping device that tracks which constraints influence the optimum. Solving the reformulated problem gives

$$\mathbf{w}=\sum_i\alpha_i y_i\Phi(\mathbf{x}_i),\qquad \sum_i\alpha_i y_i=0.$$

Thus, instead of treating $\mathbf{w}$ as an independent vector, the dual expresses it as a weighted combination of the mapped training examples. Substituting this expression into the primal objective gives

$$\begin{aligned}
\lVert\mathbf{w}\rVert^2
&=\left\langle\sum_i\alpha_i y_i\Phi(\mathbf{x}_i),\sum_j\alpha_j y_j\Phi(\mathbf{x}_j)\right\rangle \\[1mm]
&=\sum_{i,j}\alpha_i\alpha_jy_i y_j\left\langle\Phi(\mathbf{x}_i),\Phi(\mathbf{x}_j)\right\rangle.
\end{aligned}$$

The transformed feature vectors now appear only through pairwise inner products. We collect these comparisons in the **kernel matrix**, also called the **Gram matrix**:

$$K[i,j]=K(\mathbf{x}_i,\mathbf{x}_j)=\left\langle\Phi(\mathbf{x}_i),\Phi(\mathbf{x}_j)\right\rangle.$$

For a linear SVM, $\Phi(\mathbf{x})=\mathbf{x}$, so $K[i,j]=\mathbf{x}_i\cdot\mathbf{x}_j$. Using this matrix, the hard-margin dual is

$$\boxed{\max_{\boldsymbol{\alpha}}\;\sum_i\alpha_i-\frac12\sum_{i,j}\alpha_i\alpha_jy_i y_jK[i,j]\quad\text{subject to}\quad\alpha_i\geq0,\;\sum_i\alpha_i y_i=0.}$$

The primal and dual are two descriptions of the same fitted SVM. Examples with $\alpha_i=0$ do not contribute to the fitted score. Examples with $\alpha_i>0$ are the **support vectors** used by the model. The value $\alpha_i$ is a coefficient in the fitted model—not a class probability or a general feature-importance score. For a soft-margin SVM, the dual adds the bound $0\leq\alpha_i\leq C$.

The dual is especially useful because it needs only the pairwise comparisons $K[i,j]$, rather than the transformed feature vectors themselves. This creates the bridge to kernel methods: a classical or quantum kernel can compute these comparisons, and the SVM can use them to fit the same maximum-margin classifier.



### From Dot Products to Kernel Functions

A **kernel function** supplies the feature-space comparisons required by the dual:

$$K(\mathbf{x}_i,\mathbf{x}_j)=\langle\Phi(\mathbf{x}_i),\Phi(\mathbf{x}_j)\rangle.$$

We can therefore train a linear SVM in the transformed feature space without explicitly constructing $\Phi(\mathbf{x})$. This is the **kernel trick**. A useful kernel exposes a representation where a simple, preferably large-margin separator has low error. For every finite input set, a valid Gram matrix must be symmetric and positive semidefinite: $\mathbf{c}^\top K\mathbf{c}\geq0$ for every real vector $\mathbf{c}$.

Concentric rings make the idea visible. No straight line separates them in the original 2D coordinates. Define the explicit scalar feature $h(\mathbf{x})=\lVert\mathbf{x}\rVert^2$, then compare radii with a one-dimensional RBF kernel:

$$K(\mathbf{x}_i,\mathbf{x}_j)=\exp\!\left[-\gamma\bigl(h(\mathbf{x}_i)-h(\mathbf{x}_j)\bigr)^2\right].$$

For well-separated rings and a suitable $\gamma$, similar radii receive large kernel values and different radii receive smaller ones. This construction is an RBF kernel applied after an explicit radial transform; Section 3 will also test the standard RBF kernel directly on the 2D coordinates.

![Concentric rings mapped by squared radius to separated one-dimensional groups and a block-structured kernel matrix](images/svm_kernel_reveals_structure.svg)

*The matrix contains pairwise comparisons, not predictions: the SVM still combines it with the training labels.*



### How Support Vectors Make a Prediction

We began with the linear SVM score

$$g(\mathbf{x})=\mathbf{w}\cdot\mathbf{x}+b.$$

For a linear SVM, the dual formulation expresses the learned weight vector as

$$\mathbf{w}=\sum_i\alpha_i y_i\mathbf{x}_i.$$

Substituting this expression into the score gives

$$g(\mathbf{x})=\sum_i\alpha_i y_i\left(\mathbf{x}_i\cdot\mathbf{x}\right)+b.$$

A kernel SVM replaces the dot product with the kernel comparison introduced above:

$$g(\mathbf{x})=\sum_i\alpha_i y_iK(\mathbf{x}_i,\mathbf{x})+b.$$

Only support vectors have $\alpha_i>0$. All other training examples have $\alpha_i=0$ and contribute nothing, so prediction can be written as

$$\boxed{g(\mathbf{x})=\sum_{i\in\mathrm{SV}}\alpha_i y_iK(\mathbf{x}_i,\mathbf{x})+b,\qquad \hat y=\operatorname{sign}\!\bigl(g(\mathbf{x})\bigr).}$$

In plain language, the model compares the new input with each support vector. It weights those comparisons using the learned coefficients $\alpha_i$ and class labels $y_i$, adds $b$, and uses the sign of the result as the predicted class.

This explains why support vectors matter: they are the training examples that continue to determine the decision boundary after training. The kernel—classical or quantum—defines how examples are compared; the SVM determines how those comparisons are combined into a prediction.



### Putting the Pieces Together

We now have a foundation to understand the below workflow.  We will first explore the importance of kernel choices for classical models, and then extend to quantum SVM kernels in the following sections.

![Four-stage SVM workflow from labeled data through a kernel matrix and fitted support vectors to prediction](images/svm_workflow.svg)



## 3. Classical SVM Baseline on a Small Dataset

From this point onward, the exercises use soft-margin `SVC` with its default $C=1$. Its score threshold and slack variables are handled by the solver. The models differ in how their kernels compare inputs, while the same soft-margin training procedure chooses the boundary.

### Shared Dataset: Concentric Circles

The **concentric circles** dataset contains an inner ring and an outer ring, so no single straight line can separate the two classes. The **training set** is the portion used to fit each SVM. The **test set** is held aside until evaluation; test accuracy measures how often the fitted model correctly classifies these unseen examples and is one indicator of **generalization**.

Training points use circles in the plot below, while test points use triangles. We **standardize** each feature by subtracting the training-set mean and dividing by the training-set standard deviation. The scaler is fitted only on training data and then reused unchanged on test data, preventing information from the test set from leaking into training.

In [ ]:
np.random.seed(42)

# Shared classical dataset: concentric circles.
def get_small_dataset(n_samples=42, noise=0.2, factor=0.5, test_size=0.3):
    X, y = make_circles(
        n_samples=n_samples,
        noise=noise,
        factor=factor,
        random_state=42,
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = get_small_dataset()
CLASS_COLORS = {0: "#6F42C1", 1: "#76B900"}  # purple and NVIDIA green
print("Train size:", len(y_train), "Test size:", len(y_test), "Features: 2")

In [ ]:
# Scatter: train (circles) and test (triangles), colored by class
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], c=CLASS_COLORS[0], marker="o", s=60, edgecolors="white", label="Class 0")
ax.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c=CLASS_COLORS[1], marker="o", s=60, edgecolors="white", label="Class 1")
ax.scatter(X_test[y_test==0, 0], X_test[y_test==0, 1], c=CLASS_COLORS[0], marker="^", s=80, edgecolors="black", alpha=0.9)
ax.scatter(X_test[y_test==1, 0], X_test[y_test==1, 1], c=CLASS_COLORS[1], marker="^", s=80, edgecolors="black", alpha=0.9)
ax.set_xlabel("Feature 0"); ax.set_ylabel("Feature 1")
ax.set_title("Concentric circles (○ train, △ test)"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

### Compare Linear, Polynomial, and RBF Kernels

We now train three classical SVMs—each with a different classical SVM kernel—on the same 2D concentric-circles dataset and compare their decision boundaries. This makes the effect of SVM kernel choice concrete while holding the data split and regularization setting fixed.

**Mathematical definitions of the three classical SVM kernels**

For input vectors $\mathbf{x}_i, \mathbf{x}_j \in \mathbb{R}^n$, the SVM kernel function $K(\mathbf{x}_i,\mathbf{x}_j)$ measures their similarity in a chosen feature space.

#### Linear kernel

$$
K(\mathbf{x}_i,\mathbf{x}_j)=\mathbf{x}_i^\top\mathbf{x}_j
$$

This is the ordinary inner product in the original feature space. Its decision boundary is a hyperplane—in two dimensions, a straight line.

#### Polynomial kernel

For polynomial degree $p$, scale $\gamma$, and constant $c_0$,

$$
K(\mathbf{x}_i,\mathbf{x}_j)=\left(\gamma\,\mathbf{x}_i^\top\mathbf{x}_j+c_0\right)^p.
$$

Scikit-learn calls $c_0$ `coef0`. When $c_0=0$, only degree-$p$ monomials appear; a nonzero value also introduces lower-degree terms. We use $p=2$ because a quadratic boundary is naturally aligned with concentric-circle geometry.

#### Radial basis function (RBF) kernel

$$
K(\mathbf{x}_i,\mathbf{x}_j)=\exp\!\left(-\gamma\lVert\mathbf{x}_i-\mathbf{x}_j\rVert^2\right).
$$

Here $\gamma$ is an inverse squared length scale: if the Gaussian width is $\sigma$, then $\gamma=1/(2\sigma^2)$. Larger $\gamma$ gives each training point a narrower region of influence. Together with $C$ and the data, it controls how flexible the smooth nonlinear boundary can become.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 1: Compare Classical SVM Kernels</span>**

Before running the comparison, make two predictions:

- Which kernel should work best for concentric circles? Explain your choice using the boundary shapes described above.
- Where do you expect the support vectors to appear relative to each model's decision boundary?

The next cell fits linear, degree-2 polynomial, and RBF soft-margin SVMs to the same standardized data with $C=1$, then reports their training and test accuracies. The figure that follows shades each model's predicted regions, overlays the held-out test examples as triangles, and rings the training examples that became support vectors.

After viewing the results, answer:

1. Which kernel achieved the highest test accuracy, and did it match your prediction?
2. How do the locations of the ringed support vectors relate to each fitted boundary and its mistakes?

</div>

In [ ]:
# Train classical SVMs with three SVM kernel choices (same data, scaled features)
svc_linear = SVC(kernel="linear").fit(X_train, y_train)
svc_poly   = SVC(kernel="poly", degree=2).fit(X_train, y_train)
svc_rbf    = SVC(kernel="rbf").fit(X_train, y_train)
svc_classical = svc_rbf  # keep for later comparison with QSVM

results_cl = [
    ("Linear",      svc_linear.score(X_train, y_train), svc_linear.score(X_test, y_test)),
    ("Polynomial",  svc_poly.score(X_train, y_train),   svc_poly.score(X_test, y_test)),
    ("RBF",         svc_rbf.score(X_train, y_train),  svc_rbf.score(X_test, y_test)),
]
print("Classical SVM — kernel comparison")
print("-" * 42)
for name, tr, te in results_cl:
    print(f"  {name:12s}  train: {tr:.2%}   test: {te:.2%}")
acc_train_cl, acc_test_cl = results_cl[2][1], results_cl[2][2]  # RBF for later

**Why SVM kernel choice matters—decision regions and support vectors:** The plotting function creates a grid of new inputs, predicts their labels, and shades the resulting regions. It then overlays the training examples as circles and the held-out test examples as triangles. A black ring identifies each training example that the fitted model retained as a support vector.

Compare the three panels rather than focusing on the raw number of support vectors. Their locations show which examples constrain each fitted boundary, while the test triangles show how well that boundary generalizes. Purple represents class 0 and green represents class 1.

In [ ]:
def plot_svm_boundary(ax, X_train, y_train, X_test, y_test, model, title,
                      grid_n=25, feature_map=None, X_train_angles=None,
                      scale_to_angles_fn=None):
    """Plot decision boundary for any SVM (classical or quantum).

    For classical SVM: call with just model (kernel='linear'/'rbf'/etc.)
    For quantum SVM:   also pass feature_map, X_train_angles, scale_to_angles_fn
    The ONLY difference is how grid predictions are obtained.
    """
    # Step 1: build a grid of fake points (identical for both)
    x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
    y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_n), np.linspace(y_min, y_max, grid_n))
    grid_points = np.c_[xx.ravel(), yy.ravel()]

    # Step 2: predict every grid point — THIS is the only line that differs
    if feature_map is not None:
        grid_angles = scale_to_angles_fn(grid_points)
        K_grid = build_kernel_matrix(grid_angles, X_train_angles, feature_map)
        Z = model.predict(K_grid)
    else:
        Z = model.predict(grid_points)

    # Step 3: shade the predicted regions and draw the decision boundary
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5],
                colors=[CLASS_COLORS[0], CLASS_COLORS[1]], alpha=0.16)
    ax.contour(xx, yy, Z, levels=[0.5], colors="#111827", linewidths=2)

    # Step 4: overlay training circles and held-out test triangles
    for class_id, color in CLASS_COLORS.items():
        train_mask = y_train == class_id
        test_mask = y_test == class_id
        ax.scatter(X_train[train_mask, 0], X_train[train_mask, 1],
                   c=color, marker="o", s=52, edgecolors="white")
        ax.scatter(X_test[test_mask, 0], X_test[test_mask, 1],
                   c=color, marker="^", s=82, edgecolors="black")

    # Step 5: ring the training examples retained as support vectors
    support_points = X_train[model.support_]
    ax.scatter(support_points[:, 0], support_points[:, 1], s=145,
               facecolors="none", edgecolors="#111827", linewidths=2,
               label="Support vector")
    ax.legend(loc="upper right", fontsize=8, frameon=True)
    ax.set_xlabel("Feature 0"); ax.set_ylabel("Feature 1"); ax.set_title(title)

# Classical: model.predict(raw_points) — kernel computed internally by formula
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, (name, model) in zip(axes, [("Linear", svc_linear), ("Polynomial (deg=2)", svc_poly), ("RBF", svc_rbf)]):
    plot_svm_boundary(ax, X_train, y_train, X_test, y_test, model, title=name, grid_n=100)
plt.tight_layout(); plt.show()

## 4. From a Classical SVM Kernel to a CUDA-Q QSVM

Section 3 showed that changing the kernel changes the comparisons available to an SVM and can produce very different decision boundaries. We now keep the same classification problem and soft-margin SVM, but replace the classical kernel formula with similarities computed from quantum states. This lets us ask whether a quantum circuit provides a useful representation of the data. It does not assume that a quantum kernel will be more accurate than the classical kernels; that must be evaluated on held-out data.

The QSVM used here is therefore a **hybrid kernel method**, not a quantum replacement for the SVM optimizer. The change is how the SVM kernel entries are evaluated: a quantum circuit now supplies the pairwise similarities.

A **quantum feature map** is a fixed circuit $U(\mathbf{x})$ that encodes a classical input as $\ket{\phi(\mathbf{x})}=U(\mathbf{x})\ket{0}$. We compare two inputs using the **state fidelity**

$$
K_Q(\mathbf{x}_i,\mathbf{x}_j)=\left\lvert\langle\phi(\mathbf{x}_i)\vert\phi(\mathbf{x}_j)\rangle\right\rvert^2
$$

which is 1 for identical states and 0 for orthogonal states. This is a valid positive-semidefinite kernel because it can be written as the inner product $\operatorname{Tr}[\rho(\mathbf{x}_i)\rho(\mathbf{x}_j)]$ between density-matrix features $\rho(\mathbf{x})=\ket{\phi(\mathbf{x})}\bra{\phi(\mathbf{x})}$.

The classical kernels in Section 3 were evaluated internally from formulas. Here we explicitly build the kernel matrices first:

1. **Encode:** map each classical input to circuit angles and prepare its quantum state.
2. **Compare:** calculate the fidelity for each pair of training states to build $K_{\mathrm{train}}$.
3. **Fit:** give $K_{\mathrm{train}}$ and the training labels to `SVC(kernel="precomputed")`. Here, `precomputed` means that the SVM receives a completed table of kernel values instead of evaluating its own kernel formula.
4. **Predict:** build the test-to-training matrix $K_{\mathrm{test}}$ and apply the fitted SVM.

In this notebook, a simulator calculates the state overlaps deterministically. Quantum hardware would instead estimate an overlap probability from repeated measurements. This example demonstrates how a quantum kernel enters the SVM pipeline.

We use the simplest two-qubit angle encoding. For the angle vector $\mathbf{a}=(a_0,a_1)$, the fixed feature-map circuit prepares

$$
\ket{\phi(\mathbf{a})}=\bigl(R_y(a_0)\otimes R_y(a_1)\bigr)\ket{00},
$$

and the quantum fidelity kernel is

$$
K_Q(\mathbf{a},\mathbf{b})=\lvert\langle\phi(\mathbf{a})\vert\phi(\mathbf{b})\rangle\rvert^2.
$$

The circuit has no trainable parameters. It fixes the similarity measure; the classical SVM then learns the coefficients and offset that define the classifier. Exercise 2 builds this complete pipeline on the same concentric-circles data used in Section 3.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 2: Build a Quantum-Kernel SVM</span>**

Follow one quantum kernel from classical inputs to final predictions using the standardized concentric-circles data from Section 3. Complete Parts A and B; Part C then provides the short scikit-learn step that uses your matrices:

- **Part A — Encode:** map each input feature to an angle in $[0,\pi]$ and define the two-qubit CUDA-Q feature map.
- **Part B — Build and validate:** construct $K_{\mathrm{train}}$ and $K_{\mathrm{test}}$, inspect their shapes, and verify the mathematical properties required of the training kernel matrix.
- **Part C — Fit and evaluate:** use the provided SVM code to fit and evaluate a classifier from the completed matrices.

</div>

#### Part A — Map the Data and Define the Circuit

The classical SVMs used standardized coordinates directly. Rotation gates require angles, so fit a per-feature affine map using **only the training set**. Reuse that fitted map unchanged for the test set and, later, the decision-boundary grid. This avoids test-data leakage. Complete the transformer so that each training feature spans $[0,\pi]$; clip transformed test values to that same interval.

In [ ]:
cudaq.set_target("nvidia")
print(f"CUDA-Q version: {cudaq.__version__}")
print(f"Section 4 target: {cudaq.get_target().name}")


def fit_angle_transformer(X_train, angle_max=np.pi):
    """Fit and return a per-feature angle map using training data only."""
    
    mins = X_train.min(axis=0)
    ranges = X_train.max(axis=0) - mins
    ranges[ranges < 1e-10] = 1.0
# 
    def transform(X):
        # TODO START
        raise NotImplementedError("Complete this exercise TODO.")
        # TODO END
    
    return transform
    

circles_to_angles = fit_angle_transformer(X_train, angle_max=np.pi)
X_train_angles = circles_to_angles(X_train)
X_test_angles = circles_to_angles(X_test)
print("Encoded features: 2 (→ 2 qubits), angle range: [0, π]")
print("\nTraining inputs before and after angle mapping:")
print("  standardized x0   standardized x1   angle a0   angle a1")
print(np.column_stack((X_train, X_train_angles)).round(3))

Next, implement the circuit $U(\mathbf{a})=R_y(a_0)\otimes R_y(a_1)$. One qubit represents each of the two input features. The resulting kernel entry is the squared overlap between the two states prepared by this circuit; there is no separate trainable quantum model.

**What this first example is—and is not:** With only independent $R_y$ rotations, the two-qubit state remains separable. If $u(a)=[\cos(a/2),\sin(a/2)]$, then the circuit represents the explicit feature vector $\phi(\mathbf{a})=u(a_0)\otimes u(a_1)$, and

$$
K_Q(\mathbf{a},\mathbf{b})=\lvert\phi(\mathbf{a})\cdot\phi(\mathbf{b})\rvert^2
=\prod_{j=0}^{1}\cos^2\!\left(\frac{a_j-b_j}{2}\right).
$$

In other words, this rotation-only example uses the quantum computer to evaluate a squared dot-product similarity that is also straightforward to compute classically. Its purpose is to show how a quantum-computed kernel enters the SVM workflow, not to provide a quantum advantage. For this family of feature maps to have any chance of a quantum advantage, the circuit must introduce nonclassical multi-feature structure through entanglement placed so that it actually changes the data-dependent comparison—for example, by re-uploading data after entangling gates. Section 5 explores that construction. Entanglement is necessary for that possibility here, but it is not sufficient by itself to guarantee either better accuracy or a computational advantage.

In [ ]:
# EXERCISE 2A
@cudaq.kernel
def feature_map_2q(a0: float, a1: float):
    """Encode two feature values as single-qubit rotation angles."""
    q = cudaq.qvector(2)
    # TODO START
    raise NotImplementedError("Complete this exercise TODO.")
    # TODO END

#### Part B — Build and Validate the Kernel Matrices

`cudaq.get_state` returns the state prepared by the feature map. For two such states, `state_i.overlap(state_j)` evaluates $\langle\phi(\mathbf{a}_i)|\phi(\mathbf{a}_j)\rangle$. Complete `quantum_kernel_entry` so it returns the squared magnitude of that overlap. The supplied matrix builder caches the states and uses symmetry to avoid repeating training-pair calculations.

For $N$ training examples and $M$ test examples, build and inspect two matrices:

- $K_{\mathrm{train}}$ has shape $N\times N$: every training example compared with every training example.
- $K_{\mathrm{test}}$ has shape $M\times N$: every test example compared with every training example.

Before building them, predict their numerical shapes for the current dataset. Then write assertions that verify $K_{\mathrm{train}}$ is symmetric, has a unit diagonal, contains values in $[0,1]$ up to numerical tolerance, and is positive semidefinite. These checks apply to the square training matrix; $K_{\mathrm{test}}$ is rectangular and need not be symmetric.

In [ ]:
# EXERCISE 2B

def quantum_kernel_entry(state_i, state_j):
    """Compute k(x_i, x_j) = |<phi(x_i)|phi(x_j)>|^2 using state overlap."""
    # TODO START
    raise NotImplementedError("Complete this exercise TODO.")
    # TODO END

def build_kernel_matrix(X_rows, X_cols, feature_map):
    """Build a deterministic state-vector fidelity kernel matrix."""
    n_rows, n_cols = len(X_rows), len(X_cols)
    K = np.zeros((n_rows, n_cols))
    row_states = [cudaq.get_state(feature_map, float(X_rows[i][0]), float(X_rows[i][1])) for i in range(n_rows)]
    symmetric = n_rows == n_cols and np.array_equal(X_rows, X_cols)
    col_states = row_states if symmetric else [
        cudaq.get_state(feature_map, float(X_cols[j][0]), float(X_cols[j][1]))
        for j in range(n_cols)
    ]
    for i in range(n_rows):
        start = i if symmetric else 0
        for j in range(start, n_cols):
            K[i, j] = quantum_kernel_entry(row_states[i], col_states[j])
            if symmetric:
                K[j, i] = K[i, j]
    return K

In [ ]:
# Build the two matrices.
# TODO START
raise NotImplementedError("Complete this exercise TODO.")
# TODO END

print("K_train shape:", K_train.shape)
print("K_test shape: ", K_test.shape)
eigenvalues = np.linalg.eigvalsh(K_train)
numerical_tolerance = 1e-5
print(f"K_train value range: [{K_train.min():.9f}, {K_train.max():.9f}]")
print("Smallest K_train eigenvalue:", eigenvalues.min())

# Assertions for the five required properties.
# 1. K_train has shape (number of training examples, number of training examples).
# 2. K_test has shape (number of test examples, number of training examples).
# 3. K_train is symmetric and has a unit diagonal.
# 4. Every K_train entry lies in [0, 1], allowing a small numerical tolerance.
# 5. K_train is positive semidefinite up to floating-point tolerance.
# TODO START
raise NotImplementedError("Complete this exercise TODO.")
# TODO END

#### Part C — Insert the Quantum Kernel into the SVM Workflow

The quantum work is now complete: $K_{\mathrm{train}}$ and $K_{\mathrm{test}}$ contain all similarities needed by the SVM. The provided code performs three classical steps:

1. `SVC(kernel="precomputed")` tells scikit-learn that we will supply kernel matrices instead of a built-in kernel formula.
2. `fit(K_train, y_train)` learns the support-vector coefficients and offset from the training comparisons and labels.
3. `score` uses $K_{\mathrm{train}}$ or $K_{\mathrm{test}}$ to compute the fraction of correct predictions. Each row of $K_{\mathrm{test}}$ compares one test example with every training example.

The optimizer, learned support vectors, and prediction rule are classical—the quantum circuit supplied only the kernel values.

In [ ]:
# Fit a classical SVM using the quantum-computed kernel matrices.
svc_qsvm = SVC(kernel="precomputed")
svc_qsvm.fit(K_train, y_train)

acc_train_qsvm = svc_qsvm.score(K_train, y_train)
acc_test_qsvm = svc_qsvm.score(K_test, y_test)
print("QSVM (two-qubit fidelity kernel)")
print(f"  Train accuracy: {acc_train_qsvm:.2%}")
print(f"  Test accuracy:  {acc_test_qsvm:.2%}")

### Inspect the Quantum Fidelity SVM Kernel

The heatmap shows $K[i,j]$ for the training points. Rows and columns are deliberately reordered using the known class labels, so this is a **supervised visualization** of possible within-class blocks—not an unsupervised discovery of the classes. Block structure can indicate useful similarities, but it does not by itself demonstrate generalization; the held-out test accuracy is the relevant check.

In [ ]:
# Heatmap of K_train: reorder rows/cols by label so class 0 and class 1 blocks are visible
order = np.argsort(y_train)
K_sorted = K_train[np.ix_(order, order)]
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
im = ax.imshow(K_sorted, cmap="viridis", aspect="auto", vmin=0, vmax=1)
ax.set_title("Quantum fidelity SVM kernel matrix (train, reordered by class)")
ax.set_xlabel("Training point (sorted by label)"); ax.set_ylabel("Training point (sorted by label)")
plt.colorbar(im, ax=ax, label=r"$k(x_i, x_j)$"); plt.tight_layout(); plt.show()

### Visualize the QSVM Decision Boundary

The next plot uses the fitted precomputed-kernel SVM. Each grid point is first transformed to quantum angles, compared with the training points through the quantum fidelity SVM kernel, and then classified. Compare the resulting boundary with the three classical boundaries from Section 3.

In [ ]:
# Quantum: model.predict(K_grid) — kernel precomputed from quantum states
# Uses the SAME plot_svm_boundary function; only difference is passing the feature map
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
plot_svm_boundary(ax, X_train, y_train, X_test, y_test, svc_qsvm, "QSVM (2-qubit angle encoding)",
                  grid_n=20, feature_map=feature_map_2q, X_train_angles=X_train_angles,
                  scale_to_angles_fn=circles_to_angles)
plt.tight_layout(); plt.show()

## 5. Comparing Quantum Feature Maps

Two input features are not enough to expose many differences between circuit designs, so this section introduces three hidden **latent variables** $z_0,z_1,z_2$. Each latent variable generates two noisy observed features, and the label depends on products among the latent variables.

- $x_0,x_1$ are noisy observations of $z_0$.
- $x_2,x_3$ are noisy observations of $z_1$.
- $x_4,x_5$ are noisy observations of $z_2$.

The experiment holds the dataset, angle scaling, SVM optimizer, and $C=1$ fixed while changing only the quantum feature map. This isolates the question: does adding data re-uploading and entangling connectivity produce a kernel that generalizes better on this interaction-dependent dataset?

If the same data-independent unitary $V$ is the final operation in every circuit, then it cancels from the fidelity:

$$
\left\lvert\bra{0}U^\dagger(y)V^\dagger VU(x)\ket{0}\right\rvert^2
=
\left\lvert\bra{0}U^\dagger(y)U(x)\ket{0}\right\rvert^2.
$$

**Data re-uploading** avoids this cancellation when new data-dependent gates follow the entangling layer. The tested ladder is:

- **Angle-only:** one $R_y$ encoding layer and no entanglement.
- **Nearest-neighbor re-uploading:** $R_y\rightarrow$ nearest-neighbor CNOTs $\rightarrow R_y$.
- **All-to-all re-uploading:** two encoding layers, all-pairs CNOTs, and a final data-dependent $R_z$ layer.

In [ ]:
cudaq.set_target("nvidia")
print(f"Section 5 target: {cudaq.get_target().name}")

# ---------- 1. Correlated dataset (6 features, interaction-dependent labels) ----------
# 3 latent variables drive 6 observed features in highly correlated pairs.
# The label depends on ALL pairwise products of the latent variables,
# including an interaction that is nonlocal in the chosen qubit layout.

n_features_complex = 6

def get_complex_dataset(n_samples=300, test_size=0.3):
    """Dataset with highly correlated features and interaction-dependent labels."""
    rng = np.random.RandomState(42)

    # 3 latent variables → 6 features (pairs share a latent source)
    latent = rng.randn(n_samples, 3)
    noise = 0.2
    X = np.column_stack([
        latent[:, 0] + noise * rng.randn(n_samples),  # feature 0 ← z0
        latent[:, 0] + noise * rng.randn(n_samples),  # feature 1 ← z0 (corr ≈ 0.96)
        latent[:, 1] + noise * rng.randn(n_samples),  # feature 2 ← z1
        latent[:, 1] + noise * rng.randn(n_samples),  # feature 3 ← z1 (corr ≈ 0.96)
        latent[:, 2] + noise * rng.randn(n_samples),  # feature 4 ← z2
        latent[:, 2] + noise * rng.randn(n_samples),  # feature 5 ← z2 (corr ≈ 0.96)
    ])

    # Label depends on ALL pairwise products of latent variables:
    # z0*z1 — adjacent groups (qubits 0-1 vs 2-3)
    # z1*z2 — adjacent groups (qubits 2-3 vs 4-5)
    # z0*z2 — distant groups (qubits 0-1 vs 4-5)
    signal = latent[:, 0] * latent[:, 1] + latent[:, 1] * latent[:, 2] + latent[:, 0] * latent[:, 2]
    y = (signal > 0).astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_test  = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test

X_train_c, X_test_c, y_train_c, y_test_c = get_complex_dataset(n_samples=300)

# Per-feature scaling to [0, pi/2] — empirically mitigates concentration here.
_mins = X_train_c.min(axis=0)
_maxs = X_train_c.max(axis=0)
_ranges = _maxs - _mins
_ranges[_ranges < 1e-10] = 1.0
angle_max = np.pi / 2
X_train_c_angles = angle_max * (X_train_c - _mins) / _ranges
X_test_c_angles  = angle_max * np.clip((X_test_c - _mins) / _ranges, 0, 1)

corr_matrix = np.abs(np.corrcoef(X_train_c.T))
mean_abs_corr = (corr_matrix.sum() - n_features_complex) / (n_features_complex * (n_features_complex - 1))

print(f"Correlated dataset: {len(y_train_c)} train, {len(y_test_c)} test, "
      f"{n_features_complex} features → {n_features_complex} qubits")
print(f"Mean |correlation|: {mean_abs_corr:.3f}  (within-pair ≈ 0.96)")
print(f"Angle range: [0, π/2]   Class balance: {y_train_c.mean():.0%} class 1")

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 3: Does More Entanglement Improve the Kernel?</span>**

Complete three six-qubit feature maps that induce three quantum fidelity kernels:

1. **Angle-only:** apply one $R_y(x_i)$ encoding layer with no two-qubit gates.
2. **Nearest-neighbor re-uploading:** encode with $R_y$, apply a nearest-neighbor CNOT chain, and encode the data again with $R_y$.
3. **All-to-all re-uploading:** use two $R_y$ encoding stages each followed by an all-pairs CNOT block (two CNOT blocks total), and a final data-dependent $R_z$ rotation layer.

Before running the comparison, predict whether test accuracy should increase monotonically as the circuits become more connected. The supplied code builds the train and test fidelity matrices for each map and fits the same precomputed-kernel SVM.

After running it, answer:

1. Which feature map has the highest test accuracy? Does performance improve monotonically with additional entangling gates?
2. Does any map achieve high training accuracy without a comparable test improvement? What would that suggest?
3. How does the best quantum kernel compare with the classical RBF baseline?
4. What can—and cannot—be concluded from this one dataset split?

</div>

In [ ]:
# ---------- 2. Feature-map kernels ----------
# A final data-independent CNOT layer cancels exactly from a fidelity kernel.
# DATA RE-UPLOADING avoids that cancellation: encode, entangle, then encode
# again. The ladder below increases both depth and connectivity so that we can
# compare the resulting kernels empirically:
#
#   Angle-only  → 1 encoding layer, no entanglement
#   NN re-uploading  → encode, nearest-neighbour CNOTs, then re-encode
#   All-to-all  → 2 encoding layers, all-pairs CNOTs + R_z phase

@cudaq.kernel
def fmap_angle(x: list[float]):
    """Angle-only: single R_y layer — product state, no entanglement."""
    q = cudaq.qvector(n_features_complex)
    # TODO START
    raise NotImplementedError("Complete this exercise TODO.")
    # TODO END

@cudaq.kernel
def fmap_nn_2layer(x: list[float]):
    """NN re-uploading: R_y → NN CNOTs → R_y."""
    q = cudaq.qvector(n_features_complex)
    # TODO START
    raise NotImplementedError("Complete this exercise TODO.")
    # TODO END


@cudaq.kernel
def fmap_full_2layer(x: list[float]):
    """All-to-all re-uploading with two R_y/CNOT stages and a final R_z stage."""
    q = cudaq.qvector(n_features_complex)
    # TODO START
    raise NotImplementedError("Complete this exercise TODO.")
    # TODO END

### Build and Evaluate Each Quantum Fidelity SVM Kernel

The provided state-vector builder caches one state per input, constructs the symmetric training matrix, and constructs the rectangular test matrix. The evaluation cell then fits the same classical `SVC(kernel="precomputed")` for every feature map. Because the data split, preprocessing, and $C=1$ remain fixed, the comparison isolates the feature-map choice.

In [ ]:
def build_kernel_matrix_sv(X_rows, X_cols, fmap_kernel):
    """Build a deterministic state-vector fidelity kernel matrix."""
    n_rows, n_cols = len(X_rows), len(X_cols)
    K = np.zeros((n_rows, n_cols))
    row_states = [cudaq.get_state(fmap_kernel, list(X_rows[i])) for i in range(n_rows)]
    symmetric = n_rows == n_cols and np.array_equal(X_rows, X_cols)
    col_states = row_states if symmetric else [
        cudaq.get_state(fmap_kernel, list(X_cols[j])) for j in range(n_cols)
    ]
    for i in range(n_rows):
        start = i if symmetric else 0
        for j in range(start, n_cols):
            K[i, j] = abs(row_states[i].overlap(col_states[j]))**2
            if symmetric:
                K[j, i] = K[i, j]
    return K

In [ ]:
# ---------- 3. Train & test with each quantum fidelity SVM kernel ----------
kernels = {
    "Angle-only":       fmap_angle,
    "NN re-uploading":   fmap_nn_2layer,
    "All-to-all re-uploading": fmap_full_2layer,
}

print(f"\nQSVM — feature map ladder ({n_features_complex} qubits, "
      f"{len(y_train_c)} train, {len(y_test_c)} test, state-vector overlap)")
print("=" * 70)

results = {}
for name, fmap_fn in kernels.items():
    t0 = time.time()
    K_tr = build_kernel_matrix_sv(X_train_c_angles, X_train_c_angles, fmap_fn)
    K_te = build_kernel_matrix_sv(X_test_c_angles, X_train_c_angles, fmap_fn)
    svc = SVC(kernel="precomputed").fit(K_tr, y_train_c)
    train_acc = svc.score(K_tr, y_train_c)
    test_acc  = svc.score(K_te, y_test_c)
    elapsed   = time.time() - t0
    results[name] = (train_acc, test_acc, elapsed)
    print(f"  {name:25s}  train {train_acc:.2%}  test {test_acc:.2%}  ({elapsed:.1f}s)")

# Classical RBF baseline
svc_rbf = SVC(kernel="rbf").fit(X_train_c, y_train_c)
print(f"  {'Classical RBF':25s}  train {svc_rbf.score(X_train_c, y_train_c):.2%}  "
      f"test {svc_rbf.score(X_test_c, y_test_c):.2%}")
print("=" * 70)

**Interpreting the results:** The table compares feature maps on the same six-feature dataset, whose label depends on pairwise products of latent variables ($z_0z_1+z_1z_2+z_0z_2$).

The angle-only map produces a factorized similarity. The re-uploading maps place data-dependent rotations on both sides of entangling gates, allowing the fidelity to depend on cross-feature structure. However, the three circuits do not form a guaranteed accuracy ordering. Extra connectivity may add useful structure, irrelevant structure, or flexibility that fits the training set without improving held-out predictions.

Use the test accuracies—not circuit size or training accuracy alone—to answer Exercise 3. The outcome is evidence for this dataset and split, not a general ranking of quantum feature maps.

### Before Exercise 4: write your feature-map specification

Exercise 4 asks an AI agent to search for a quantum feature map under a gate budget. Before opening any AI tool, write your answers to these questions in a new markdown cell:

1. **Success criterion:** What test accuracy on the 6-feature correlated dataset would you accept as meaningfully better than the angle-only baseline? Write the number.
2. **Validity check:** What mathematical property must the kernel matrix satisfy for the SVM to be well-defined? How will you verify that the agent's feature map satisfies it?
3. **Classical simulability:** How will you determine whether the agent's proposed circuit is classically simulable — i.e., whether its fidelity kernel could be reproduced without a quantum computer?
4. **Gate budget convention:** The exercise allows up to 30 single-qubit gates and 15 two-qubit gates. Which of these will you treat as your binding constraint, and why?

**Takeaway:** Entanglement changes a fidelity kernel only when its placement changes the data-dependent state comparison. Greater connectivity is a modeling choice, not a guarantee of better generalization; compare feature maps using held-out data under a controlled experimental setup.

### Extension Exercise: Autoresearch a Feature Map Under a Gate Budget

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 4: Agent-Guided Quantum Feature-Map Search</span>**

Inspired by [Andrej Karpathy's **autoresearch**](https://github.com/karpathy/autoresearch), use an iterative experiment loop to search for a quantum feature-map ansatz instead of choosing one circuit by hand. The problem is deliberately open ended: you must make and justify the research choices that are not fixed below.

**Goal:** Find the quantum feature-map ansatz that gives the best held-out test-set accuracy while respecting the gate budget. This models a setting in which QPU execution time, rather than classical model fitting, is the limiting resource.

**1. Choose the problem.** Find an existing binary-classification dataset or generate a reproducible synthetic one with **15–20 numerical input features** and substantial feature correlation. To make "highly correlated" testable, every retained feature must have $|r| \geq 0.70$ with at least one other retained feature, where $r$ is the Pearson correlation computed on the training split. Explain why the classification target is nontrivial, cite the source or document the generator and random seed, and show the training-set correlation matrix. Choose and justify the sample size and a fixed stratified train/validation/test split. Fit every preprocessing operation on the training split only.

**2. Respect the circuit budget.** A candidate state-preparation circuit $U(\mathbf{x})$ may use at most **30 logical single-qubit gates** and **15 logical two-qubit gates**. This mimics having the QPU as the limited resource.

**3. Run a reproducible search.** Define a finite search budget before starting (for example, at most 15 candidate ansätze). For every candidate, save: (i) a circuit description or diagram, (ii) exact single- and two-qubit gate counts for $U(\mathbf{x})$, (iii) train and validation accuracy, (iv) kernel-matrix checks, and (v) kernel construction and fitting time. Use validation accuracy to guide the search and choose one final ansatz; break ties by fewer two-qubit gates, then fewer single-qubit gates, then shorter runtime. Do not inspect test labels or test accuracy while designing or selecting candidates.

**4. Compare and report.** At minimum, compare the selected ansatz with an angle-only quantum kernel and a classical RBF SVM. You may also include the nearest-neighbor and all-to-all re-uploading ansätze from Exercise 3 as baselines. Compute and report the exact single- and two-qubit gate counts for every quantum baseline; clearly mark any baseline that exceeds the 30/15 budget as **outside budget** and therefore ineligible to win. After freezing all designs, evaluate them on the test split once. Report the resulting test accuracies, identify the best budget-compliant ansatz, and discuss the accuracy–gate-count tradeoff and if any interesting patterns emerged. 

</div>

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise: Audit an AI-generated feature map</span>**

Suppose an AI coding assistant generated the following feature map in response to the prompt *"Write a 6-qubit quantum feature map for an SVM kernel."*

```python
@cudaq.kernel
def fmap_ai_generated(angles: list[float]):
    q = cudaq.qvector(6)
    for i in range(6):
        ry(angles[i], q[i])
```

This code runs without error and produces a valid positive-semidefinite kernel matrix. Looking at the angle-only results in the table earlier in this section, answer the following:

1. Is this feature map classically simulable? Explain why or why not.
2. Is the accuracy of this kernel evidence of quantum advantage? Why not?
3. What is the minimum change to this circuit that would make its fidelity kernel non-trivially quantum?
4. What would you write back to the AI assistant to get a less trivial feature map?

</div>

## 6. Scaling Feature Dimension with a QPU-Style Tensor-Network QSVM

A quantum SVM has two different scaling pressures. The number of examples determines the size of the SVM kernel matrix: a symmetric matrix for $N$ training examples has $N(N+1)/2$ unique values. For a fidelity kernel, however, the $N$ diagonal values satisfy $K(\mathbf{x}_i,\mathbf{x}_i)=1$ exactly, leaving $N(N-1)/2$ nontrivial training entries to evaluate, plus one entry for every test–training pair. On a QPU, each evaluated entry generally requires many repeated circuit executions, or shots, so increasing the number of examples remains costly. A quantum computer does not remove this approximately quadratic kernel-matrix bottleneck.

The nontrivial kernel entries are independent, so a platform with multiple QPUs can evaluate them asynchronously across devices. CUDA-Q's MQPU backend can similarly distribute independent evaluations across multiple GPUs acting as simulated QPUs, provided each individual circuit fits on one GPU. This can reduce wall-clock time, although it does not reduce the total number of entries or shots. Refactoring a sequential kernel-matrix builder to schedule the entries, collect the asynchronous results, and reconstruct the matrix correctly is a great task for agentic AI; NVIDIA's [CUDA-Q agent skill](https://build.nvidia.com/skills?q=cuda-q) can help inspect the relevant APIs, implement the multi-device workflow, and validate the result.

The number and structure of the **features** affect a different part of the computation: the cost and complexity of each entry $K(\mathbf{x},\mathbf{y})$. More features can require more qubits, and interaction-aware encodings can require entangling gates that represent correlations among those features. The potential role of a QPU is therefore not to make a large matrix small, but to act as a specialized processor for its entries. The most plausible regime for this workflow is feature-rich but data-limited: a modest number of examples keeps the matrix manageable while each example contains structured, high-dimensional information. Whether a quantum kernel is actually useful still depends on the dataset, feature map, hardware cost, and comparison with strong classical kernels.

This section demonstrates that separation using handwritten 0s and 1s. We crop each 8×8 image to its informative central 6×6 region, giving 36 correlated pixel features and a 36-qubit entangling circuit for every kernel entry. This removes mostly blank border pixels while retaining the visual structure of the original image. A dense complex64 state vector would require 512 GiB, far beyond the lesson GPU. Instead, the `tensornet` backend represents the circuit as an exact tensor network—with no MPS truncation—and obtains measurement samples by contracting that network. Its cost still depends on circuit depth, entanglement, contraction structure, and shot count; tensor networks do not make arbitrary quantum circuits easy.

Earlier sections prepared two state vectors and computed their overlap. Here we use the equivalent overlap-circuit identity

$$K_Q(\mathbf{x},\mathbf{y})=\lvert\braket{\phi(\mathbf{y})}{\phi(\mathbf{x})}\rvert^2=\lvert\braket{0}{U^\dagger(\mathbf{y})U(\mathbf{x})\ket{0}}\rvert^2.$$

On a QPU we cannot request or inspect the state vector. We must execute $U^\dagger(\mathbf{y})U(\mathbf{x})$, measure every qubit, and estimate the kernel entry from the all-zero outcome frequency. Section 6 uses only this QPU-compatible circuit-and-measurement interface, with no simulator state access. The tensor-network backend simulates the circuit, while finite-shot sampling makes the estimated kernel entries statistical. The small classification result is a backend-scaling demonstration, not evidence of model generalization or quantum advantage.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 5: Scaling Quantum-Kernel Construction Under a Runtime Budget</span>**

Complete the three parts of this QPU-style kernel exercise:

**a. State-vector limit.** Calculate the memory required to store a 36-qubit dense state vector using complex64 amplitudes and using complex128 amplitudes. Run the next cell to check your calculation.

**b. Overlap circuit.** Complete the CUDA-Q kernel below. It must prepare $U(\mathbf{x})$, apply $U^\dagger(\mathbf{y})$ manually in reverse gate order, and measure every qubit. Do not construct, return, or inspect a state vector.

**c. Sampled SVM kernel.** Within an approximate five-minute execution budget, estimate every nontrivial entry as the fraction of shots that returns the all-zero bit string. Set the training diagonal to its exact value, $K(\mathbf{x}_i,\mathbf{x}_i)=1$, rather than executing redundant self-overlap circuits. For the supplied 16-training/4-test split, first verify that this requires 120 unique off-diagonal training evaluations and 64 test–training evaluations, or 184 total. Predict the runtime from the supplied per-entry reference, then build both complete matrices, compare the prediction with the measured runtime, inspect their numerical properties, and fit the precomputed-kernel SVM. Explain why increasing the number of examples increases the number of pairwise entries—and therefore total QPU work—whereas increasing the feature count changes the circuit used for each entry.

</div>

In [ ]:
# EXERCISE 5
n_qubits = 36
for dtype_name, bytes_per_amplitude in [("complex64", 8), ("complex128", 16)]:
    gibibytes = 2**n_qubits * bytes_per_amplitude / 2**30
    tebibytes = gibibytes / 1024
    print(f"{dtype_name}: {gibibytes:,.0f} GiB ({tebibytes:.1f} TiB)")

# Even 36 qubits exceed the 6 GiB memory of the lesson GPU by almost two orders of magnitude.

### A Visual 36-Feature Dataset with a Larger Kernel Matrix

We use sklearn's well-known Optical Recognition of Handwritten Digits dataset—not 28×28 MNIST—and retain only digits 0 and 1. The images make the classification problem easy to inspect visually. The outer border is mostly background for these centered digits, so we crop each 8×8 image to its central 6×6 region. This gives 36 informative, correlated pixel features rather than spending qubits on nearly constant border pixels. We order those pixels along a continuous snake path and entangle consecutive qubits, allowing the kernel to depend on local image relationships rather than treating every pixel as isolated.

Because pixel intensities have the known physical range 0–16, the same fixed angle map is used for training, test, and future images. The maximum angle $\pi/8$ acts as a fixed kernel bandwidth; the smaller range keeps different 36-qubit image states sufficiently close that their all-zero probabilities can be resolved with a modest number of shots. We use 16 training and 4 test images. The backend samples the 120 unique off-diagonal entries of the symmetric $16\times16$ training matrix and every cell of the $4\times16$ test matrix. The known training diagonal is filled with ones without circuit execution, leaving exactly 184 sampled entries. At 30 shots per entry, this is 5,520 circuit shots. A measured reference time of about 1.3 seconds per entry places the construction near four minutes on the lesson's RTX A1000; runtime will vary with hardware and CUDA-Q version. Four test images are still far too few for a generalization claim—the learning objective is to expose the cost and construction of a nontrivial kernel matrix within a bounded lab runtime.

In [ ]:
# ---------- Load sklearn digits (8x8) — binary: digit 0 vs digit 1 ----------
digits = load_digits()
mask = np.isin(digits.target, [0, 1])
X_digit_images = digits.images[mask]            # (360, 8, 8)
X_digits = X_digit_images[:, 1:7, 1:7].reshape(-1, 36)
y_digits = digits.target[mask]                   # 0 or 1

(X_d_train_raw, X_d_test_raw, X_d_train_images, X_d_test_images,
 y_d_train, y_d_test) = train_test_split(
    X_digits, X_digit_images, y_digits,
    test_size=0.3, stratify=y_digits, random_state=42
)

# Quantum encoding: pixels have a known domain [0, 16]. Use that fixed map
# for train, test, and future data; no split-dependent min/max fitting.
digit_angle_max = np.pi / 8
X_d_train_angles = digit_angle_max * X_d_train_raw / 16.0
X_d_test_angles  = digit_angle_max * X_d_test_raw / 16.0

# Traverse alternate image rows in opposite directions. Consecutive
# features now follow a continuous nearest-neighbour path through the image.
snake_indices = np.array([
    row * 6 + col
    for row in range(6)
    for col in (range(6) if row % 2 == 0 else range(5, -1, -1))
])
X_d_train_angles = X_d_train_angles[:, snake_indices]
X_d_test_angles = X_d_test_angles[:, snake_indices]

# Execution-budgeted demonstration: 16×15/2 off-diagonal training entries
# plus all 4×16 test–training entries gives 184 circuit evaluations.
train_sub_d, test_sub_d = 16, 4

def balanced_subset_indices(y, n, seed=42):
    """Choose a deterministic, approximately class-balanced subset."""
    rng = np.random.RandomState(seed)
    classes = np.unique(y)
    counts = [n // len(classes)] * len(classes)
    for i in range(n % len(classes)):
        counts[i] += 1
    chosen = [rng.choice(np.flatnonzero(y == cls), size=count, replace=False)
              for cls, count in zip(classes, counts)]
    indices = np.concatenate(chosen)
    rng.shuffle(indices)
    return indices

idx_train_d = balanced_subset_indices(y_d_train, train_sub_d, seed=42)
idx_test_d  = balanced_subset_indices(y_d_test, test_sub_d, seed=43)
X_t_d, X_te_d = X_d_train_angles[idx_train_d], X_d_test_angles[idx_test_d]
y_t_d, y_te_d = y_d_train[idx_train_d], y_d_test[idx_test_d]

# Keep raw pixel values for plotting later.
X_te_d_images = X_d_test_images[idx_test_d]

n_entries = (
    train_sub_d * (train_sub_d - 1) // 2 + test_sub_d * train_sub_d
)
print(f"Digits dataset: {len(y_digits)} samples of digits 0 and 1")
print("Quantum input: central 6×6 crop → 36 features and 36 qubits")
print(f"Class split: {(y_digits==0).sum()} zeros, {(y_digits==1).sum()} ones")
print(f"Using balanced subset: {train_sub_d} train, {test_sub_d} test")
print(f"Fixed pixel encoding: [0, 16] → [0, π/8]")
print(f"Kernel entries that require sampling: {n_entries}")

# fp64 is the default. We choose fp32 to reduce numerical storage; this
# lesson does not claim that fp32 is faster for every circuit or GPU.
tn_precision = "fp32"
shots_per_entry = 30
cudaq.set_target("tensornet", option=tn_precision)
print(f"Tensor-network precision: {tn_precision}")
print(f"Shots per kernel entry: {shots_per_entry}")
print(f"Matrix-construction circuit shots: {n_entries * shots_per_entry}")

### Part B — Write the QPU-Compatible Overlap Circuit

Complete `overlap_digits_snake` without using any state-vector API. The input arrays have already been reordered along the image's snake path. Implement these operations in order:

- Allocate 36 qubits.
- Prepare $U(\mathbf{x})$: apply $R_y(x_i)$ to every qubit, apply CZ gates along the 35 consecutive snake-path pairs, then re-upload the data with $R_x(x_i/2)$.
- Apply $U^\dagger(\mathbf{y})$: negate every angle and reverse the layer order. Reverse the CZ loop as well, even though these CZ gates commute, to practice constructing a general adjoint circuit.
- Measure all 36 qubits with `mz(q)`.

For identical inputs, the circuit reduces to the identity and must return the all-zero bit string in the noiseless simulator. For different inputs, the probability of that bit string is the quantum fidelity SVM kernel value. The data-dependent rotations between the two CZ layers prevent the entangling operations from simply cancelling, so the kernel is not a product of 36 independent one-pixel similarities. The connected chain is the shallowest CZ topology that links every feature; a two-dimensional grid would better match the image geometry but would be substantially more expensive to contract.

In [ ]:
n_digit_qubits = 36

@cudaq.kernel
def overlap_digits_snake(x: list[float], y: list[float]):
    """Prepare, uncompute, and measure U†(y)U(x)|0>."""
    # TODO START
    raise NotImplementedError("Complete this exercise TODO.")
    # TODO END

### Part C — Estimate the Kernel Entries from Measurements

`cudaq.sample` is the only execution primitive used below. It runs the completed overlap circuit for a finite number of shots and returns measurement counts, just as a QPU would. The estimate

$$\widehat K_Q(\mathbf{x},\mathbf{y})=\frac{\text{all-zero counts}}{\text{shots}}$$

has statistical uncertainty. Increasing the shot count improves precision but increases runtime on both the simulator and a QPU. The exact `tensornet` backend means the underlying simulated circuit has no MPS truncation; it does not make the finite-shot estimate exact. For a symmetric training matrix, the builder sets the known diagonal $K(\mathbf{x}_i,\mathbf{x}_i)=1$, samples only the strict upper triangle, and copies each measured value into the corresponding lower-triangle position. It therefore executes neither self-overlap circuits nor reversed duplicates. It also uses CUDA-Q's [parameter broadcasting](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.sample) to submit all required input pairs in one `sample` call, reducing Python and launch overhead.

The snake circuit is a structured, shallow compromise. It is entangling and follows local image relationships, but its one-dimensional connectivity remains tractable enough for this sampling demonstration. A deeper or two-dimensional circuit can represent additional correlations while making tensor-network contraction, QPU execution, and noise more expensive.

There is no physical-QPU instruction that returns an exact basis-state amplitude. CUDA-Q's documented execution choices are measurement sampling, kernel returns, observable expectation values, or simulator state access. The last option can retrieve a selected amplitude efficiently in simulation, but it is intentionally excluded here because it cannot run on a QPU. With multiple QPUs or GPUs, [`sample_async`](https://nvidia.github.io/cuda-quantum/latest/using/examples/executing_kernels.html#sample-asynchronous) can distribute independent entries; on one GPU, asynchronous submission does not remove the sequential device work.

### Build the Training and Test Kernel Matrices

The following cell builds both complete matrices using 184 circuit evaluations. The training calculation sets all 16 diagonal entries to one and samples the 120 entries strictly above the diagonal. Each measured off-diagonal value is copied into its mirrored location, so neither self-overlaps nor reversed pairs are executed. The test calculation samples all 64 test–training entries because that rectangular matrix has no corresponding symmetry to exploit. The same code structure can target a QPU because it depends only on circuit execution and measurement counts, not simulator state access.

The unit diagonal follows analytically from $U^\dagger(\mathbf{x})U(\mathbf{x})=I$; it is not an empirical check in this construction. Interpret the timings—which include first-call compilation—as measurements for this circuit, backend, and GPU, not universal tensor-network performance numbers.

In [ ]:
all_zeros_digits = "0" * n_digit_qubits

def build_sampled_kernel_matrix(
    X_rows, X_cols, overlap_kernel, shots, symmetric=False
):
    """Estimate a fidelity-kernel matrix from all-zero frequencies."""
    n_rows, n_cols = len(X_rows), len(X_cols)
    if symmetric and (n_rows != n_cols or not np.array_equal(X_rows, X_cols)):
        raise ValueError("symmetric=True requires identical row and column data")
    K = np.eye(n_rows) if symmetric else np.zeros((n_rows, n_cols))
    pairs = ([(i, j) for i in range(n_rows) for j in range(i + 1, n_cols)]
             if symmetric else
             [(i, j) for i in range(n_rows) for j in range(n_cols)])
    if not pairs:
        return K, 0.0

    # Broadcast all required pairs in one sample call to reduce host overhead.
    X_batch = np.asarray([X_rows[i] for i, _ in pairs])
    Y_batch = np.asarray([X_cols[j] for _, j in pairs])
    start = time.perf_counter()
    batch_counts = cudaq.sample(
        overlap_kernel, X_batch, Y_batch, shots_count=shots
    )
    for (i, j), counts in zip(pairs, batch_counts):
        K[i, j] = counts.count(all_zeros_digits) / shots
        if symmetric:
            K[j, i] = K[i, j]
    return K, time.perf_counter() - start

# Select the CUDA-Q backend explicitly; no separate tensor-network API is used.
cudaq.set_target("tensornet", option=tn_precision)
print(f"Simulation target: {cudaq.get_target().name} "
      f"({tn_precision.upper()})")

X_train_tn, y_train_tn = X_t_d, y_t_d
X_test_tn, y_test_tn = X_te_d, y_te_d
tn_train_n, tn_test_n = len(y_train_tn), len(y_test_tn)

K_train_tn, train_seconds = build_sampled_kernel_matrix(
    X_train_tn, X_train_tn, overlap_digits_snake,
    shots=shots_per_entry, symmetric=True)
K_test_tn, test_seconds = build_sampled_kernel_matrix(
    X_test_tn, X_train_tn, overlap_digits_snake,
    shots=shots_per_entry)
n_tn_entries = tn_train_n * (tn_train_n - 1) // 2 + tn_test_n * tn_train_n
total_tn_time = train_seconds + test_seconds
min_eigenvalue = np.linalg.eigvalsh(K_train_tn).min()

print(f"CUDA-Q sampled all {n_tn_entries} nontrivial kernel entries")
print(f"Total: {total_tn_time:.2f}s ({total_tn_time / n_tn_entries:.3f}s/entry)")
print(f"Shots per entry: {shots_per_entry}")
print(f"Kernel symmetry error: {np.max(np.abs(K_train_tn-K_train_tn.T)):.2e}")
print("Kernel diagonal: set analytically to 1 (no circuit executions)")
print(f"Smallest sampled-kernel eigenvalue: {min_eigenvalue:.3e}")
assert np.allclose(np.diag(K_train_tn), 1.0)

### Fit the SVM—and Keep the Claim in Scope

Once $K$ has been estimated, fitting `SVC(kernel="precomputed")` is entirely classical and unchanged from the smaller examples. Finite-shot noise can make a sampled Gram matrix slightly non-positive-semidefinite, which is why the previous cell reports its smallest eigenvalue. The final model uses 16 training and 4 test images. Its predictions confirm that the complete measurement-based workflow runs with a 36-qubit connected feature map and a 184-entry execution workload, but the test sample remains far too small to compare model quality reliably. The result demonstrates how tensor networks let us prototype feature-rich, QPU-compatible quantum kernel entries beyond the dense state-vector limit; it does not show that the chosen kernel outperforms a classical method or that a QPU would provide an end-to-end advantage.

In [ ]:
svc_tn = SVC(kernel="precomputed").fit(K_train_tn, y_train_tn)
y_pred_tn = svc_tn.predict(K_test_tn)
print(f"Entangling CUDA-Q demo accuracy: train {svc_tn.score(K_train_tn, y_train_tn):.2%}, "
      f"test {svc_tn.score(K_test_tn, y_test_tn):.2%}")
print("Accuracy uses 16 train / 4 test images; interpret the backend workflow, not generalization.")

fig, axes = plt.subplots(1, tn_test_n, figsize=(8, 2.2))
for idx, ax in enumerate(np.atleast_1d(axes)):
    ax.imshow(X_te_d_images[idx].reshape(8, 8), cmap="gray_r")
    correct = y_pred_tn[idx] == y_test_tn[idx]
    ax.set_title(f"true={y_test_tn[idx]} pred={int(y_pred_tn[idx])}",
                 color="#76b900" if correct else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Research Connection — QSVM Simulation at 784 Qubits

[Chen *et al.* (2025)](https://doi.org/10.1088/2632-2153/adb4ba) scaled a related quantum-kernel workflow to 28×28-pixel MNIST inputs, mapping 784 features to 784 qubits. cuQuantum converted each kernel circuit into a tensor network, and cuTensorNet computed the all-zero amplitude on NVIDIA A100 GPUs without constructing the full state vector. Reusing the optimized contraction path across data pairs enabled a 784-qubit contraction in under 0.2 seconds. The authors also distributed independent kernel entries across multiple GPUs, showing how tensor networks and GPU parallelism can support much larger QSVM studies.

---

## Conclusion

### Key Takeaways

- An SVM learns a maximum-margin decision boundary, while its SVM kernel function determines the feature space in which that boundary is constructed. Changing the SVM kernel can therefore change which patterns the classifier can separate.
- A QSVM keeps the classical SVM optimization and replaces the classical similarity function with entries computed by a quantum feature-map circuit. No circuit parameters were trained in this notebook.
- Quantum feature-map design matters. Entanglement, connectivity, data re-uploading, angle scaling, and preprocessing must be tested; adding more gates does not automatically produce a better SVM kernel.
- Simulator state overlaps provide convenient, deterministic SVM kernel values for small circuits. An overlap circuit followed by measurement provides the QPU-compatible alternative when the state vector cannot be accessed.
- The number of examples and the number of features create different scaling costs. More examples enlarge the SVM kernel matrix approximately quadratically, while more features increase the qubit count and circuit complexity required to evaluate each entry.
- Exact tensor-network simulation can evaluate structured circuits beyond the dense state-vector memory limit. Its cost still depends strongly on circuit depth, connectivity, and entanglement.


**Related Notebooks:**
* [Introduction to Hybrid Quantum Neural Networks](https://github.com/NVIDIA/cuda-q-academic/blob/main/quantum-machine-learning-and-data-analysis/01_an_introduction_to_hybrid_quantum_neural_networks.ipynb) — builds a trainable hybrid QML model.
* [Advanced Hybrid Quantum Neural Networks](https://github.com/NVIDIA/cuda-q-academic/blob/main/quantum-machine-learning-and-data-analysis/02_advanced_hqnns.ipynb) — examines HQNN performance and scaling.
* [Quantum PageRank](https://github.com/NVIDIA/cuda-q-academic/blob/main/quantum-machine-learning-and-data-analysis/04_quantum_pagerank.ipynb) — applies quantum graph inference using stochastic walks.
